# Steering Validation

This notebook verifies that `ExpertSteerer` is mechanistically working as intended — not just that model outputs differ, but that the routing hooks are actually firing correctly at the gate level.

## Candidates

The experts validated here are the **Stage 1 candidates**: the (layer, expert) pairs identified by Stage 1's Risk Difference profiling as most strongly associated with a given behaviour. Specifically, an expert qualifies as a candidate only if it ranks in the top-`CANDIDATE_N` by |RD| on **both** the frequency metric and the logit metric independently for its layer — the dual-metric intersection filter from `stage1/analysis/03_candidate_selection.ipynb`. These are the exact experts suppressed in Stage 2, so validating them here confirms the intervention is mechanistically sound before interpreting behavioural results.

## Conditions

Data is produced by running `validate_steering.py` on the cluster, which passes N=200 prompts through the model under three conditions per axis:

- **Baseline** — no intervention, natural routing
- **Hard** — targeted experts are completely blocked: a forward pre-hook modifies the gate's hidden-state input `h` before routing runs, driving each suppressed expert's logit to a large negative target (τ = −10⁴) via a per-token perturbation `δh = (τ − hWᵀ) (WWᵀ)⁻¹W`; the gate then executes its native grouped top-k routing on the modified input and never selects the suppressed experts
- **Soft** — targeted experts have their gate logit shifted by `strength × RD score` before routing, via a precomputed constant hidden-state perturbation `δh`; whether an expert is displaced from the top-k depends on its routing margin

## Prompt Format

The prompts used for each axis **exactly match the experimental condition** in which the target experts were identified — because expert routing rates are input-dependent, and we need to measure suppression under the same activation regime as the actual experiment:

| Axis | Direction | Prompt format | Mirrors task |
|---|---|---|---|
| Safety | Negative (suppress compliance) | AdvBench harmful instruction → chat template → forced prefix `"Sure! Here is..."` appended | `safety_safe` |
| Safety | Positive (suppress refusal) | AdvBench harmful instruction wrapped with safety system prompt | `safety_unsafe` |
| Faithfulness | Negative (suppress confabulation) | FaithEval-Counterfactual record: `Context: ... Question: ... Options: ...` | `faith_cf` |

Only the `negative` direction is validated for faithfulness. Both directions are validated for safety.

In [1]:
import json
import os
import pandas as pd

RESULT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'stage2', 'validation_result.json')
if not os.path.exists(RESULT_PATH):
    RESULT_PATH = '../validation_result.json'

with open(RESULT_PATH) as f:
    data = json.load(f)

cfg = data['config']
print(f"candidate_n={cfg['candidate_n']}  soft_strength={cfg['soft_strength']}  n={cfg['n']}")

candidate_n=3  soft_strength=0.5  n=200


## Column Definitions

Each row is one targeted (layer, expert) pair. Expert indices are **per-layer** — expert 35 at layer 2 and expert 35 at layer 18 are completely different components; the index is just a slot number within that layer's expert pool.

| Column | What it means |
|---|---|
| **Baseline Rate** | Fraction of top-k routing slots won by this expert across all probe tokens, with no intervention. E.g. `0.044` means it was selected in 4.4% of all token-layer routing decisions. |
| **Hard Rate** | Same fraction under hard deactivation. Must be `0.000` — the hook replaces this expert every time it would have been selected. |
| **Hard OK** | `True` if hard rate is exactly zero. This is the primary mechanistic check. |
| **Soft Rate** | Same fraction under soft suppression. Should be lower than baseline — the expert's gate logit is shifted down, so it wins less often, but can still win if it's sufficiently dominant. |
| **Soft Rate Reduced** | `True` if soft rate < baseline rate. |
| **Expected Logit Shift** | The logit shift we asked for: `soft_strength × RD score` for this expert. Negative means we're pushing the logit down. |
| **Actual Logit Shift** | The logit shift measured directly from the gate inputs during the soft run. Should closely match expected — if it does, the hidden-state perturbation (`δh`) is landing correctly. |

In [2]:
def build_df(axis_data):
    rows = []
    for layer_str, layer_val in axis_data['layers'].items():
        for expert_str, e in layer_val['experts'].items():
            rows.append({
                'Layer':                int(layer_str),
                'Expert':               int(expert_str),
                'Baseline Rate':        e['baseline_rate'],
                'Hard Rate':            e['hard_rate'],
                'Hard OK':              e['hard_ok'],
                'Soft Rate':            e['soft_rate'],
                'Soft Rate Reduced':    e['soft_rate_reduced'],
                'Expected Logit Shift': e['expected_logit_shift'],
                'Actual Logit Shift':   e['actual_logit_shift'],
            })
    return pd.DataFrame(rows).sort_values(['Layer', 'Expert']).reset_index(drop=True)


def style_df(df):
    def colour_hard_rate(val):
        return 'background-color: #c8f7c5; font-weight: bold' if val == 0.0 else 'background-color: #f7c5c5; font-weight: bold'

    def colour_bool(val):
        return 'color: green; font-weight: bold' if val else 'color: red; font-weight: bold'

    def colour_shift_diff(row):
        styles = [''] * len(row)
        exp = row['Expected Logit Shift']
        act = row['Actual Logit Shift']
        if exp is not None and exp != 0:
            rel_err = abs((act - exp) / exp)
            col = '#c8f7c5' if rel_err < 0.15 else '#fff3cd' if rel_err < 0.40 else '#f7c5c5'
            styles[list(row.index).index('Actual Logit Shift')] = f'background-color: {col}'
        return styles

    styler = (
        df.style
        .map(colour_hard_rate, subset=['Hard Rate'])
        .map(colour_bool,      subset=['Hard OK', 'Soft Rate Reduced'])
        .apply(colour_shift_diff,   axis=1)
        .format({
            'Baseline Rate':        '{:.4f}',
            'Hard Rate':            '{:.4f}',
            'Soft Rate':            '{:.4f}',
            'Expected Logit Shift': '{:+.4f}',
            'Actual Logit Shift':   '{:+.4f}',
        })
    )

    if hasattr(styler, 'hide'):
        return styler.hide(axis='index')
    return styler.hide_index()

## Safety — Negative Direction (suppress compliance experts)

These are the Stage 1 candidates most associated with unsafe compliance — experts that fire preferentially when the model is producing harmful output. Validated on AdvBench prompts with forced prefix (39 candidates across 24 layers).

In [3]:
safety_neg_df = build_df(data['safety_negative'])
display(style_df(safety_neg_df))

Layer,Expert,Baseline Rate,Hard Rate,Hard OK,Soft Rate,Soft Rate Reduced,Expected Logit Shift,Actual Logit Shift
2,35,0.0373,0.0000,True,0.0072,True,-0.9727,-0.9727
3,22,0.0431,0.0000,True,0.0224,True,-1.1989,-1.1739
3,42,0.0463,0.0000,True,0.0143,True,-0.9538,-0.9548
4,41,0.0493,0.0000,True,0.0288,True,-0.7858,-0.8456
5,39,0.0477,0.0000,True,0.0182,True,-0.9847,-1.0529
6,50,0.0487,0.0000,True,0.0311,True,-0.7580,-0.8192
7,8,0.0461,0.0000,True,0.0143,True,-0.7684,-0.7519
8,43,0.0071,0.0000,True,0.0000,True,-0.6967,-0.6487
9,1,0.0298,0.0000,True,0.0000,True,-1.0393,-1.1397
9,41,0.0617,0.0000,True,0.0390,True,-0.6553,-0.7016


### What this validation measures — and what it does not

**What the validation ingests.** Each forward pass processes only the input sequence: the AdvBench prompt plus the forced prefix (for the negative direction) or the safety system prompt (for the positive direction). No output tokens are present. The model performs a single prefill pass and stops.

**What the actual experiment involves.** In Stage 2, `model.generate` is called with `use_cache=False`. At each generation step `t`, the model performs a full forward pass over the entire sequence so far: `[prompt + prefix + token_1 + ... + token_t]`. By step 50 of a harmful completion, the hidden states at every layer are conditioned on 50 tokens of generated output — fundamentally different in character from prefill hidden states.

This validation therefore does not observe the hidden states that occur during generation. That is a deliberate and justified choice.

**Hard deactivation: correctness is h-independent by construction.** The hard pre-hook reads the gate's input hidden state `h` to compute current logits for suppressed experts, then derives a per-token perturbation `δh = (τ − hWᵀ)(WWᵀ)⁻¹W` that drives those logits to τ = −10⁴. This computation runs afresh at every forward call — no state is cached between steps. Whether `h` encodes a prefill token conditioned only on the prompt or a generation token conditioned on fifty previously generated tokens, the hook executes the same conditional logic and guarantees the same result: suppressed experts' logits reach τ and are never selected. Demonstrating a routing rate of 0.000 across 200 diverse prefill inputs is therefore a complete proof that the suppression logic functions correctly; the same code path runs identically on every subsequent forward call during generation.

**The soft logit shift is h-independent.** The soft intervention adds a fixed precomputed vector `δh` to the hidden state before the gate runs. The resulting logit shift for expert `i` is:

&emsp; `Δlogit_i = F.linear(δh, W)[i] = δh · W[i]`

This depends only on `δh` and the gate weight `W[i]` — not on `h`. Adding `δh` shifts expert `i`'s logit by the same fixed amount on every forward call, regardless of sequence position or generation step. The *Expected* and *Actual Logit Shift* columns are therefore valid for every forward pass in a generation run, not just for prefill tokens.

**What generation-based validation would add.** The one quantity prefill validation does not capture is the *baseline routing rate during generation* — how frequently each expert is selected while the model is producing output tokens. A generation-based validation would yield more ecologically grounded routing rate estimates. However, since the correctness of both the hard suppression guarantee and the soft logit shift is provably independent of `h`, those rates carry no additional information about whether the steering mechanism is functioning correctly.

## Safety — Positive Direction (suppress refusal experts)

These are the Stage 1 candidates most associated with safe refusal — experts that fire preferentially when the model is declining a harmful request. Validated on AdvBench prompts with safety system prompt (46 candidates across 24 layers).

In [4]:
safety_pos_df = build_df(data['safety_positive'])
display(style_df(safety_pos_df))

Layer,Expert,Baseline Rate,Hard Rate,Hard OK,Soft Rate,Soft Rate Reduced,Expected Logit Shift,Actual Logit Shift
2,44,0.0173,0.0000,True,0.0120,True,-0.9843,-0.9843
3,35,0.0272,0.0000,True,0.0067,True,-0.9955,-0.9868
3,52,0.0217,0.0000,True,0.0169,True,-1.0488,-1.0142
3,63,0.0280,0.0000,True,0.0055,True,-1.1473,-1.1558
4,12,0.0213,0.0000,True,0.0065,True,-1.3592,-1.3796
4,25,0.0242,0.0000,True,0.0000,True,-1.6718,-1.6912
5,31,0.0060,0.0000,True,0.0000,True,-1.5044,-1.5140
5,49,0.0656,0.0000,True,0.0104,True,-1.3323,-1.3603
5,57,0.0378,0.0000,True,0.0167,True,-1.2421,-1.1802
6,53,0.0476,0.0000,True,0.0104,True,-1.3108,-1.3639


## Faithfulness — Negative Direction (suppress confabulation experts)

These are the Stage 1 candidates most associated with context-ignoring confabulation — experts that fire preferentially when the model ignores the provided context and generates from parametric memory. Validated on FaithEval-Counterfactual records (28 candidates across 19 layers).

In [5]:
faith_df = build_df(data['faithfulness'])
display(style_df(faith_df))

Layer,Expert,Baseline Rate,Hard Rate,Hard OK,Soft Rate,Soft Rate Reduced,Expected Logit Shift,Actual Logit Shift
1,17,0.0078,0.0000,True,0.0000,True,-1.0313,-1.0313
4,36,0.0452,0.0000,True,0.0274,True,-0.8739,-0.8810
5,7,0.0254,0.0000,True,0.0103,True,-1.1031,-1.0390
7,31,0.0552,0.0000,True,0.0255,True,-1.1323,-1.1837
8,51,0.0308,0.0000,True,0.0078,True,-1.1385,-1.1521
8,56,0.0484,0.0000,True,0.0155,True,-1.2023,-1.2350
9,27,0.0395,0.0000,True,0.0089,True,-1.1972,-1.2678
9,63,0.0346,0.0000,True,0.0069,True,-1.2681,-1.2803
10,15,0.0308,0.0000,True,0.0070,True,-1.2587,-1.2720
10,28,0.0476,0.0000,True,0.0052,True,-1.2047,-1.1984


### Note on prefill validity for faithfulness

The same h-independence argument from the safety section applies here. Additionally, the faithfulness benchmarks (FaithEval-Counterfactual, MCTest) are multiple-choice tasks: the model reads a context passage and question, then outputs a single letter. The generation phase is therefore minimal — at most a few tokens — meaning the prefill of the context prompt constitutes the overwhelming majority of the computation and the routing signal. Prefill-based validation is therefore not only mechanistically sufficient but also a close approximation of the actual experimental regime.

## Summary

**Hard check** — every targeted expert must have `Hard Rate == 0.000`. If any fail, the pre-hook is not driving suppressed experts' logits below the selection threshold — either the hook is not registered, the wrong gate module is targeted, or numerical precision is insufficient.

**Soft check** — `Soft Rate` should be below `Baseline Rate` for every expert. The logit shift columns confirm the hidden-state perturbation (`δh`) is producing the intended change in gate scores: green = within 15% of expected, yellow = within 40%, red = off by more than 40%.

> **Note on logit shift discrepancies.** Minor differences between expected and actual logit shift are expected and benign. Two sources contribute:
>
> 1. **Float16 precision loss.** `δh` is computed analytically in float32 but cast to float16 to match the model's inference dtype. float16 has limited precision (~3 significant decimal digits), so the applied shift is a slightly rounded version of what was requested.
>
> 2. **Matrix inversion error amplified by shift magnitude.** `δh` is derived by inverting `WW^T`. This inversion is numerically approximate in floating point, and the resulting error scales proportionally with the size of the requested shift (`strength × RD score`). Experts with larger |RD| scores request larger shifts, so the same relative inversion error produces a larger absolute discrepancy — which is why the yellow cases tend to be the experts with the strongest RD signal, not a random subset.
>
> Neither source indicates incorrect hook behaviour. The hooks are firing as intended; the deviation is a numerical artefact of float16 inference and finite-precision matrix inversion.

In [6]:
combined = pd.concat([
    safety_neg_df.assign(Axis='Safety (negative)'),
    safety_pos_df.assign(Axis='Safety (positive)'),
    faith_df.assign(Axis='Faithfulness (negative)'),
])

n_total   = len(combined)
n_hard_ok = combined['Hard OK'].sum()
n_soft_ok = combined['Soft Rate Reduced'].sum()

print(f"Hard check : {n_hard_ok}/{n_total} experts have routing rate == 0.000")
print(f"Soft check : {n_soft_ok}/{n_total} experts show reduced routing rate vs baseline")

if n_hard_ok == n_total:
    print("\nHard steering: ALL PASS")
else:
    print("\nHard steering: FAILURES DETECTED")
    print(combined[~combined['Hard OK']][['Axis', 'Layer', 'Expert', 'Hard Rate']].to_string(index=False))

if n_soft_ok < n_total:
    print("\nSoft steering: experts NOT reduced:")
    cols = ['Axis', 'Layer', 'Expert', 'Baseline Rate', 'Soft Rate']
    print(combined[~combined['Soft Rate Reduced']][cols].to_string(index=False))

Hard check : 115/115 experts have routing rate == 0.000
Soft check : 115/115 experts show reduced routing rate vs baseline

Hard steering: ALL PASS


---

## Token-Range Validation: Question-Span-Only Steering

### Motivation

Stage 1 RD scores are computed over **question-span tokens only** — the positions corresponding to the question text in the input sequence. This was done to measure routing during the moment the model reads the question (and decides whether to use context vs. parametric memory), isolating signal from the long context passage.

Stage 2 expert suppression was originally applied to **all tokens**. This creates a mismatch: we are suppressing experts based on question-token signal, but the hook fires on every token including the context passage. Worse, the Stage 2 input always includes a context (unlike the no-context Stage 1 condition), so "parametric" experts are already naturally depressed on context tokens — suppressing them further on those tokens is redundant at best and destructive at worst.

The fix: pass `token_range=(q_start, q_end)` to `ExpertSteerer` so that hooks fire only at the positions corresponding to the question text, matching the Stage 1 measurement site exactly.

### What this section validates

For each targeted (layer, expert) pair, the validation runs three conditions with `token_range` set to the question span:

| Column | What it means |
|---|---|
| **Baseline Q/C Rate** | Routing rate on question-span (Q) and context (C) tokens with no intervention |
| **Hard Q Rate** | Rate on question tokens under hard deactivation — should be **0.000** |
| **Hard C Rate** | Rate on context tokens under hard deactivation — should be **≈ baseline** (hook did not fire) |
| **Hard Q OK** | `True` if hard Q rate == 0.0 |
| **Hard C OK** | `True` if hard C rate is within 0.005 of baseline C rate |
| **Soft Q Rate** | Rate on question tokens under soft deactivation — should be **lower** than baseline Q |
| **Soft C Rate** | Rate on context tokens under soft deactivation — should be **≈ baseline** (hook did not fire) |
| **Soft Q Reduced** | `True` if soft Q rate < baseline Q rate |
| **Soft C OK** | `True` if soft C rate is within 0.005 of baseline C rate |

In [7]:
def build_tr_df(tr_data):
    rows = []
    for layer_str, layer_val in tr_data['layers'].items():
        for expert_str, e in layer_val['experts'].items():
            rows.append({
                'Layer':           int(layer_str),
                'Expert':          int(expert_str),
                'Baseline Q Rate': e['baseline_q_rate'],
                'Baseline C Rate': e['baseline_c_rate'],
                'Hard Q Rate':     e['hard_q_rate'],
                'Hard C Rate':     e['hard_c_rate'],
                'Hard Q OK':       e['hard_q_ok'],
                'Hard C OK':       e['hard_c_ok'],
                'Soft Q Rate':     e['soft_q_rate'],
                'Soft C Rate':     e['soft_c_rate'],
                'Soft Q Reduced':  e['soft_q_reduced'],
                'Soft C OK':       e['soft_c_ok'],
                'Exp Shift':       e['expected_logit_shift'],
                'Act Shift':       e['actual_logit_shift'],
            })
    return pd.DataFrame(rows).sort_values(['Layer', 'Expert']).reset_index(drop=True)


def style_tr_df(df):
    def colour_hard_q(val):
        return 'background-color: #c8f7c5; font-weight: bold' if val == 0.0 else 'background-color: #f7c5c5; font-weight: bold'

    def colour_bool(val):
        return 'color: green; font-weight: bold' if val else 'color: red; font-weight: bold'

    return (
        df.style
        .map(colour_hard_q, subset=['Hard Q Rate'])
        .map(colour_bool, subset=['Hard Q OK', 'Hard C OK', 'Soft Q Reduced', 'Soft C OK'])
        .format({
            'Baseline Q Rate': '{:.4f}', 'Baseline C Rate': '{:.4f}',
            'Hard Q Rate':     '{:.4f}', 'Hard C Rate':     '{:.4f}',
            'Soft Q Rate':     '{:.4f}', 'Soft C Rate':     '{:.4f}',
            'Exp Shift':       '{:+.4f}', 'Act Shift':       '{:+.4f}',
        })
    )


if 'faithfulness_token_range' in data:
    tr_df = build_tr_df(data['faithfulness_token_range'])
    display(style_tr_df(tr_df))
else:
    print("faithfulness_token_range key not found — re-run validate_steering.py to generate updated results.")

,Layer,Expert,Baseline Q Rate,Baseline C Rate,Hard Q Rate,Hard C Rate,Hard Q OK,Hard C OK,Soft Q Rate,Soft C Rate,Soft Q Reduced,Soft C OK,Exp Shift,Act Shift
0,1,17,0.0077,0.0078,0.0000,0.0078,True,True,0.0000,0.0078,True,True,-1.0313,-1.0313
1,4,36,0.0434,0.0453,0.0000,0.0453,True,True,0.0223,0.0453,True,True,-0.8739,-0.8700
2,5,7,0.0158,0.0260,0.0000,0.0260,True,True,0.0056,0.0260,True,True,-1.1031,-1.0625
3,7,31,0.0518,0.0555,0.0000,0.0554,True,True,0.0189,0.0554,True,True,-1.1323,-1.1621
4,8,51,0.0289,0.0309,0.0000,0.0309,True,True,0.0036,0.0309,True,True,-1.1385,-1.1698
5,8,56,0.0414,0.0488,0.0000,0.0488,True,True,0.0053,0.0487,True,True,-1.2023,-1.2689
6,9,27,0.0374,0.0396,0.0000,0.0396,True,True,0.0058,0.0396,True,True,-1.1972,-1.3163
7,9,63,0.0281,0.0350,0.0000,0.0350,True,True,0.0019,0.0350,True,True,-1.2681,-1.2640
8,10,15,0.0264,0.0310,0.0000,0.0310,True,True,0.0016,0.0310,True,True,-1.2587,-1.3149
9,10,28,0.0478,0.0475,0.0000,0.0475,True,True,0.0027,0.0475,True,True,-1.2047,-1.2296


In [8]:
if 'faithfulness_token_range' in data:
    n_hq = tr_df['Hard Q OK'].sum()
    n_hc = tr_df['Hard C OK'].sum()
    n_sq = tr_df['Soft Q Reduced'].sum()
    n_sc = tr_df['Soft C OK'].sum()
    n_tr = len(tr_df)
    print(f"Token-range validation ({n_tr} candidates):")
    print(f"  Hard  Q suppressed (rate == 0.0):        {n_hq}/{n_tr}")
    print(f"  Hard  C unchanged  (|Δ| < 0.005):        {n_hc}/{n_tr}")
    print(f"  Soft  Q reduced vs baseline:             {n_sq}/{n_tr}")
    print(f"  Soft  C unchanged  (|Δ| < 0.005):        {n_sc}/{n_tr}")
    if n_hq == n_tr:
        print("\n  Hard Q: ALL PASS — hooks fired exclusively on question-span tokens")
    else:
        print(f"\n  Hard Q: {n_tr - n_hq} FAILURES — suppression leaked outside question span or did not fire")
    if n_hc == n_tr:
        print("  Hard C: ALL PASS — context tokens unaffected by hook")
    else:
        print(f"  Hard C: {n_tr - n_hc} context tokens changed — possible collateral effect")
else:
    print("No token-range data available.")

Token-range validation (36 candidates):
  Hard  Q suppressed (rate == 0.0):        36/36
  Hard  C unchanged  (|Δ| < 0.005):        36/36
  Soft  Q reduced vs baseline:             36/36
  Soft  C unchanged  (|Δ| < 0.005):        36/36

  Hard Q: ALL PASS — hooks fired exclusively on question-span tokens
  Hard C: ALL PASS — context tokens unaffected by hook
